In [13]:
#print("Fonction d'Ishigami :")
#sp.pprint(f)
#print("\nIntégrale sur x2 et x3 :")
#sp.pprint(sp.simplify(integrale_x2_x3))
#sp.pprint(sp.simplify(centrageSigma0))

In [14]:
import sympy as sp
from sympy import latex
# Déclaration des variables symboliques
x1, x2, x3 = sp.symbols('x1 x2 x3')
a, b = sp.symbols('a b')

In [15]:
# Définition de la fonction d'Ishigami
f = sp.sin(x1) + a * sp.sin(x2)**2 + b * x3**4 * sp.sin(x1)
#f = a*sp.sin(x1) +  sp.sin(x2)**2 + b * sp.sin(x3) * sp.sin(x1)

In [16]:
# Calcul de l'intégrale de f sur x2 et x3 dans [-pi, pi]
integralef_x3 = sp.integrate(f,(x3, -sp.pi, sp.pi))
integralef_x2_x3 = sp.integrate(integralef_x3,(x2, -sp.pi, sp.pi))
#normalisation pour l'espérance conditionnelle de 
esperance_condf = integralef_x2_x3 / (2*sp.pi)**2
# Calcul de l'intégrale de f² sur x2 et x3 dans [-pi, pi]
f2=f*f
integralef2_x3 = sp.integrate(f2,(x3, -sp.pi, sp.pi))
integralef2_x2_x3 = sp.integrate(integralef2_x3,(x2, -sp.pi, sp.pi))
esperance_condf2 = integralef2_x2_x3 / (2*sp.pi)**2

In [17]:
#Calcul de Sigma0
#construction du vecteur (esperance_condf)^2 esperance_condf espeance_condf² 
vbric_Sigma0 = sp.Matrix([esperance_condf*esperance_condf, esperance_condf, esperance_condf2])
#calcul du centrage
espvbric_Sigma0=sp.integrate(vbric_Sigma0, (x1, -sp.pi, sp.pi))/(2*sp.pi)
centrageSigma0=espvbric_Sigma0*espvbric_Sigma0.T
#calcul de la partie quadratique à intégrer
mbric_Sigma0=vbric_Sigma0*vbric_Sigma0.T
#calcul de la matrice de covariance Sigma_0
Sigma0=sp.integrate(mbric_Sigma0, (x1, -sp.pi, sp.pi))/(2*sp.pi)-centrageSigma0

In [18]:
#Calcul de Sigma_a
vbric_Sigma_a = sp.Matrix([2*esperance_condf, 1, 1])
unzeze=sp.Matrix([1, 0, 0])
matrix_de_1_et_0=unzeze*unzeze.T
covepsf=esperance_condf2-esperance_condf*esperance_condf
Sigma_a=vbric_Sigma_a*vbric_Sigma_a.T+covepsf*matrix_de_1_et_0
#print(latex(Sigma_a))

In [19]:
#Calcul de Sigma_b
vbric_Sigma_b= sp.Matrix([f, f, f2])
#calcul du centrage
espvbric_Sigma_b=sp.integrate(
    sp.integrate(vbric_Sigma_b, (x2, -sp.pi, sp.pi)),
    (x3, -sp.pi, sp.pi)
)/ (2*sp.pi)**2
#calcul de la partie quadratique à intégrer par rapport à x2 et x3
mbric_Sigma_b=vbric_Sigma_b*vbric_Sigma_b.T
Sigma_b=sp.integrate(
    sp.integrate(mbric_Sigma_b, (x2, -sp.pi, sp.pi)),
    (x3, -sp.pi, sp.pi)
)/ (2*sp.pi)**2-espvbric_Sigma_b*espvbric_Sigma_b.T
#print(latex(Sigma_b))

In [20]:
#calcul de Sigma1
# Calcul de l'intégrande produit d'Hadamard de Sigma_a et Sigma_b
Igrand_Sigma1=sp.hadamard_product(Sigma_a,Sigma_b)
#On intègre par rapport à x1
Sigma1=sp.integrate(Igrand_Sigma1, (x1, -sp.pi, sp.pi))/(2*sp.pi)
#print(latex(Sigma1))

In [21]:
#Calcul de l'indice de Sobol' et de la variance de Y
#Calcul de l'espérance de f
espf=sp.integrate(sp.integrate(
    sp.integrate(f, (x2, -sp.pi, sp.pi)),
    (x3, -sp.pi, sp.pi)
),(x1, -sp.pi, sp.pi))/(2*sp.pi)**3
#Calcul de l'espérance de f²
espf2=sp.integrate(sp.integrate(
    sp.integrate(f2, (x2, -sp.pi, sp.pi)),
    (x3, -sp.pi, sp.pi)
),(x1, -sp.pi, sp.pi))/(2*sp.pi)**3
#Calcul de la variance de Y
varY=espf2-espf*espf
#Calcul de la variance de l'espérance conditionelle
numerator_Zobol=sp.integrate(esperance_condf*esperance_condf, (x1, -sp.pi, sp.pi))/(2*sp.pi)-espf*espf
#Calcul de l'indice de Sobol'
Zobol=numerator_Zobol/varY
Zobol=sp.simplify(Zobol)
#print(latex(Zobol))

In [22]:
#Calcul de la variance asymptotique de l'estimateur de Chatterjee
v_f=(sp.Matrix([1, 2*espf*(Zobol-1), -Zobol]))/varY
#Calcul de la variance asymptotique formule finale
sigma_Zobol_hat=v_f.T*(Sigma0+Sigma1)*v_f
sigma_Zobol_hat=sp.simplify(sigma_Zobol_hat);

In [25]:
#évalution de l'indice de Sobol' et de la variance asymptotique dans le cas a=10, b=0.1
values = {a: 10., b: 1}
Zobol_num=Zobol.evalf(subs=values);
sigma_Zobol_hat_num=sigma_Zobol_hat.evalf(subs=values);
print('Dans le cas a=10. et b=1');
print('indice de Sobol vaut: ', Zobol_num);
print('variance asymptotique vaut: ',sigma_Zobol_hat_num)

Dans le cas a=10. et b=1
indice de Sobol vaut:  0.374810635179922
variance asymptotique vaut:  Matrix([[0.778810720462778]])
